# 01 – Project Tour

**Purpose:** Explore and demonstrate the new engineering foundation components implemented to support robust ML development.

This notebook demonstrates:
1. **Configuration Management**: Loading `default.yaml` and merging with overlays (`train.yaml` and `inference.yaml`)
2. **Project Metadata**: Reading version strings, Git commit head hashes, and build/run initialization timestamps
3. **Logging & Seeding Utilities**: verifying reproducibility and structured logs output
4. **Timer Utility**: Context manager performance tracking
5. **Artifact Manager**: Saving and loading mock metrics, CSV data, and visualization figures

**No ML models are loaded or evaluated in this tour notebook.**

## 0. Environment Setup

Add the repository root to `sys.path` so we can import `src` correctly.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Configuration Management

We load the baseline configuration (`default.yaml`) and merge it with the training overlay (`train.yaml`).

In [ ]:
from src.utils.config import load_config

# Load baseline default configuration
default_cfg = load_config()
print("=== Baseline Default config (Subset) ===")
print("Project name:", default_cfg["project"]["name"])
print("Default seed:", default_cfg["seed"])
print("Default model:", default_cfg["model"]["name"])

# Load training configuration overlay (merges default + train)
train_cfg = load_config("train.yaml")
print("\n=== Training config Overlay (Subset) ===")
print("Training epochs:", train_cfg["train"]["epochs"])
print("Overridden Model:", train_cfg["model"]["name"])
print("Preserved seed:", train_cfg["seed"])  # Preserved from default.yaml

## 2. Project Metadata

Retrieve version number, Git hash, and build timestamp.

In [ ]:
from src.utils.version import __version__, get_git_commit_hash, get_build_timestamp

print("Project Version:", __version__)
print("Git Commit Hash:", get_git_commit_hash())
print("Build Timestamp:", get_build_timestamp())

## 3. Logging & Seeding Utilities

In [ ]:
from src.utils.logging_utils import get_logger
from src.utils.seed import set_seed

logger = get_logger("notebook.tour")
logger.info("Initializing notebook seed...")

# Freeze randomness
set_seed(42)
logger.info("Seed frozen at 42")

## 4. Timer Utility

Evaluate execution blocks using the `Timer` context manager.

In [ ]:
import time
from src.utils.timer import Timer

with Timer("Mock Inference Run") as t:
    time.sleep(0.25)  # Simulate execution

print(f"Captured elapsed time: {t.elapsed:.4f} seconds")

## 5. Artifact Manager

Demonstrate creating directories, saving mock JSON metrics, dataframes to CSV, and exporting figures.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.artifacts import save_metrics, save_csv, save_figure
from src.utils.constants import OUTPUT_DIR

# 1. Save mock metrics
mock_metrics = {
    "accuracy": 0.892,
    "f1_weighted": 0.887,
    "inference_latency_ms": 12.4,
}
save_metrics(mock_metrics, OUTPUT_DIR / "metrics" / "mock_tour_metrics.json")

# 2. Save mock CSV dataset
df = pd.DataFrame({
    "ticket_id": [101, 102, 103],
    "category": ["Billing", "Technical", "Technical"],
    "predicted_routing": ["Finance", "IT Support", "IT Support"]
})
save_csv(df, OUTPUT_DIR / "metrics" / "mock_predictions.csv")

# 3. Save mock figure
plt.figure(figsize=(6, 3))
sns.barplot(x=df["category"].value_counts().index, y=df["category"].value_counts().values)
plt.title("Mock Prediction Categories")
fig = plt.gcf()
save_figure(fig, OUTPUT_DIR / "figures" / "mock_category_bar.png")
plt.close()

print("\nAll mock artifacts generated successfully!")